
# Megaminx GUI notebook (Colab / Jupyter)

Этот ноутбук **клонирует весь репозиторий из GitHub** `visualcomments/agents_4_puzzles`,
устанавливает зависимости и запускает **графический интерфейс** для megaminx-пайплайнов.

Через GUI можно:
- выбрать **все** или **только отдельные** пайплайны;
- проверить **все** или **только выбранные** `g4f`-модели;
- выбрать одну `g4f`-модель или набор моделей для prompt-пайплайнов;
- держать `regular` как **from scratch** вариант без baseline;
- запускать `baseline_no_llm`, `best_tested_solver`, `optimized_assets_v3_top150/top300` и все prompt-варианты;
- при наличии `kaggle.json` делать preflight и live submit.


In [ ]:

from __future__ import annotations

import shutil
import subprocess
import sys
from pathlib import Path

try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False

WORKDIR = Path('/content' if IN_COLAB else '.').resolve()
REPO_URL = 'https://github.com/visualcomments/agents_4_puzzles.git'
REPO_BRANCH = 'main'
REPO_DIR = WORKDIR / 'agents_4_puzzles'
FORCE_RECLONE = True
GIT_DEPTH = 1

INSTALL_FULL_REQUIREMENTS = True
INSTALL_G4F = 'none'        # none | github | pypi
INSTALL_CAYLEYPY = 'none'   # none | github | pypi
EXTRA_PIP_PACKAGES = []
GRADIO_SHARE = False

print({'WORKDIR': str(WORKDIR), 'REPO_URL': REPO_URL, 'BRANCH': REPO_BRANCH})


In [ ]:

def run(cmd, cwd=None):
    cwd = cwd or WORKDIR
    print('$', ' '.join(str(x) for x in cmd))
    return subprocess.run([str(x) for x in cmd], cwd=str(cwd), check=True, text=True)

if FORCE_RECLONE and REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    run(['git', 'clone', '--depth', str(GIT_DEPTH), '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], cwd=WORKDIR)
else:
    run(['git', 'fetch', 'origin', REPO_BRANCH], cwd=REPO_DIR)
    run(['git', 'checkout', REPO_BRANCH], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=REPO_DIR)

print('Repo ready:', REPO_DIR)


In [ ]:

run([sys.executable, '-m', 'pip', 'install', '-U', 'pip'], cwd=REPO_DIR)
run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'requirements-megaminx-gui.txt')], cwd=REPO_DIR)
run([sys.executable, '-m', 'pip', 'install', 'kaggle'], cwd=REPO_DIR)
if INSTALL_FULL_REQUIREMENTS and (REPO_DIR / 'requirements-full.txt').exists():
    run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'requirements-full.txt')], cwd=REPO_DIR)
if INSTALL_G4F == 'github':
    run([sys.executable, '-m', 'pip', 'install', 'git+https://github.com/xtekky/gpt4free'], cwd=REPO_DIR)
elif INSTALL_G4F == 'pypi':
    run([sys.executable, '-m', 'pip', 'install', 'g4f'], cwd=REPO_DIR)
if INSTALL_CAYLEYPY == 'github':
    run([sys.executable, '-m', 'pip', 'install', 'git+https://github.com/cayleypy/cayleypy'], cwd=REPO_DIR)
elif INSTALL_CAYLEYPY == 'pypi':
    run([sys.executable, '-m', 'pip', 'install', 'cayleypy'], cwd=REPO_DIR)
if EXTRA_PIP_PACKAGES:
    run([sys.executable, '-m', 'pip', 'install', *EXTRA_PIP_PACKAGES], cwd=REPO_DIR)


In [ ]:

import importlib.util

app_path = REPO_DIR / 'megaminx_gui_app.py'
spec = importlib.util.spec_from_file_location('megaminx_gui_app', app_path)
module = importlib.util.module_from_spec(spec)
assert spec is not None and spec.loader is not None
spec.loader.exec_module(module)

demo = module.create_demo(repo_dir=REPO_DIR)
demo.queue(default_concurrency_limit=1).launch(share=GRADIO_SHARE, inline=True)



## Что дальше

1. Вкладка **Setup** — при необходимости поставьте `g4f`, `cayleypy`, загрузите `kaggle.json`.
2. Вкладка **g4f models** — проверьте все или выбранные модели.
3. Вкладка **Pipelines** — выберите пайплайны, режим моделей и параметры `v3`.
4. Вкладка **Results** — посмотрите лучший `submission.csv` и, если нужно, отправьте его на Kaggle.
